# Влияние годичного параллакса на положения звезд

In [25]:
from math import *
import numpy as np
from datetime import datetime
from astropy.time import Time
from astroquery.jplhorizons import Horizons

### Функции, для вычисления  $X,Y,Z,\dot X,\dot Y,\dot Z$ геоцентра на основе <a href=https://www.phpsciencelabs.com/vsop87-source-code-generator-tool/ target = _blank>Multi-Language VSOP87 Source Code Generator Tool v2.2</a>

In [26]:
import os
import ctypes
_cpy = ctypes.CDLL(os.path.abspath('libem.so'))
_cpy.earthPosVel.argtypes = (ctypes.c_double,ctypes.POINTER(ctypes.c_double))
_cpy.earthPosVelEq.argtypes = (ctypes.c_double,ctypes.POINTER(ctypes.c_double))
def earth_motion_ecliptic(JD):
    global _cpy
    posvel = np.zeros(6)
    _cpy.earthPosVel(ctypes.c_double(JD),posvel.ctypes.data_as(ctypes.POINTER(ctypes.c_double)))
    return posvel
def earth_motion_equatorial(JD):
    global _cpy
    posvel = np.zeros(6)
    _cpy.earthPosVelEq(ctypes.c_double(JD),posvel.ctypes.data_as(ctypes.POINTER(ctypes.c_double)))
    return posvel
    

## Это мы тоже знаем. Наши родные тангенциальные координаты

In [27]:
def tangential_coordinates(ra,dec,RA,DEC):
    ksi = cos(dec)*sin(ra-RA)/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    eta = (sin(dec)*cos(DEC)-cos(dec)*sin(DEC)*cos(ra-RA))/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    return ksi,eta

def equatorial_coordinates(ksi,eta,RA,Dec):
    x,y,z = np.dot(np.array([[-sin(RA),-cos(RA) * sin(Dec),cos(RA) * cos(Dec)],
                             [cos(RA),-sin(RA) * sin(Dec),sin(RA) * cos(Dec)],
                             [0,cos(Dec),sin(Dec)]]),
                   np.array([ksi,eta,1]))/sqrt(1+ksi*ksi+eta*eta)
    ra = atan2(y,x)
    dec = atan2(z,sqrt(x*x+y*y))
    if(ra<0):
        ra+=2*pi
    return ra,dec

## А вот и функция для вычисления изменений координат звезды из-за параллактического смещения для звезды с параллаксом $\varpi$ для момента времени $t$

$\begin{matrix}
\xi =  \varpi(X(t)\sin\alpha_0 -Y(t)\cos\alpha_0 ) \\
\eta =  \varpi(X(t)\cos\alpha_0\sin\delta_0+ Y(t)\sin\alpha_0\sin\delta_0-Z(t)cos\delta_0)
\end{matrix}$

In [28]:
def parallactic_shift(ra,dec,JD,plx):
    e_x,e_y,e_z,e_dot_x,e_dot_y,e_dot_z = earth_motion_equatorial(JD)
    delta_ra = plx*(e_x*sin(ra)-e_y*cos(ra))
    delta_dec = plx*(e_x*cos(ra)*sin(dec)+e_y*sin(ra)*sin(dec)-e_z*cos(dec))
    return delta_ra,delta_dec

## Вычисляем смещения для разных моментов времени 

In [29]:
ra,dec = radians(0.0),radians(40.0)
plx = 0.130
JD = float(Time(datetime.utcnow()).copy(format='jd').value)
JDs = np.linspace(JD,JD+5*365.25,300)

x,y = np.zeros(np.size(JDs)),np.zeros(np.size(JDs))
for k in range(np.size(JDs)):
    x[k],y[k] = parallactic_shift(ra,dec,JDs[k],plx)

## Смотрим, как выглядит параллактическое движение в экваториальных координатах

In [30]:
import matplotlib.pyplot as plt
%matplotlib widget

fig, ax = plt.subplots(figsize=(3, 2), dpi=300, tight_layout=True)
ax.set_aspect(aspect=1)
plt.scatter(x, y, c=JDs, s=0.2)
# plt.xlim(-30,30)
# plt.ylim(-30,30)
ax.xaxis.set_tick_params(labelsize=3)
ax.yaxis.set_tick_params(labelsize=3)
plt.xlabel('$\\Delta \\alpha$, arcsec', fontsize = 4)
plt.ylabel('$\\Delta \\delta$, arcsec', fontsize = 4)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …